### 第十三章 網路爬蟲入門

#### 13.1 認識網路爬蟲

##### 13.1.1 網路爬蟲的類型與應用

##### 13.1.2 爬蟲的工作原理

##### 13.1.3 網頁的請求與回應

#### 13.2 使用 requests 發送 GET 請求

In [1]:
# ch13_1.py, 發送GET請求
import requests				# 載入套件
url = 'https://www.udn.com'	# 目標網站
req = requests.get(url) 		# 發送GET請求
if req.status_code == 200:    	# 狀態碼200 表示請求成功
    print('GET請求成功！')    	# 印出請求成功
    print(req.text[:41])		# 印出回應的HTML內容前41字
else:
    print(f'請求失敗！狀態碼:{req.status_code}')

GET請求成功！
<html><head><script language="javascript"


#### 13.3 解析 HTML 與安裝 BeautifulSoup

##### 13.3.1 簡單的 HTML 檔案

In [ ]:
<!-- cats.html, 有圖片、超連結的HTML文檔 -->
<!DOCTYPE html>
<html lang="zh-TW">
<head>
    <meta charset="UTF-8">
    <title class="about">關於貓</title>
</head>
<body>
    <h1 class="knowledge">貓咪小知識</h1>
    <div>
        <p id="p1" class="cats">
            貓咪神祕浪漫、優雅迷人。</p>
        <p id="p2" style="font-size:10pt">
            獨立自信，不受束縛。</p>
        <h2 class="pictures">貓咪圖片</h2>
        <img src="cats_QQ.jpg" alt="可愛貓咪一">
        <img src="cats_Mi.jpg" alt="可愛貓咪二">
    </div>
    <div>
        <h3 class="Related">相關連結</h3>
        <ul>
            <li><a href="https://www.catsdate.com/">
                    CatsDate</a></li>
            <li><a href="https://www.pettalk.tw/">
                    PetTalk說寵物</a></li>
        </ul>
    </div>
</body>
</html>

##### 13.3.2 認識 HTML 的標籤

##### 13.3.3 開發人員工具與網頁原始碼

##### 13.3.4 安裝與載入 BeautifulSoup

#### 13.4 實作：抓取網頁特定資訊

##### 13.4.1 取得標籤內容

In [2]:
# ch13_2.py, 取得各種標籤的內容
import requests
from bs4 import BeautifulSoup
url = 'https://wienhong.github.io/cats.html'
req = requests.get(url)
req.encoding = 'utf-8'
sp = BeautifulSoup(req.text, 'html.parser')
print(sp.title.text.strip())	# 列印<title>標籤的內容
print(sp.h1.text.strip())		# 列印<h1>標籤的內容
print(sp.p.text.strip())		# 列印<p>標籤的內容
print(sp.a.text.strip())		# 列印<a>標籤的內容

關於貓
貓咪小知識
貓咪神祕浪漫、優雅迷人。
CatsDate


##### 13.4.2 取得超連結與圖片內容

In [3]:
# ch13_3.py, 取得圖片和超連結網址
import requests
from bs4 import BeautifulSoup
url = 'https://wienhong.github.io/cats.html'
req = requests.get(url)
req.encoding = 'utf-8'
sp = BeautifulSoup(req.text, 'html.parser')

# 找出所有的<img>標籤
images = sp.find_all('img') 	# 找出所有的<img>標籤
for img in images:
    img_url = img.get('src')  	# 取得圖片的 URL
    print(f'圖片網址: {img_url}')

# 爬取並列印所有超連結的網址
links = sp.find_all('a')  		# 找到所有 <a> 標籤
for link in links:
    href = link.get('href')  	# 取得超連結的 URL
    if href:  				# 檢查是否有 href 屬性
        print(f'超連結網址: {href}')  	# 列印超連結的 URL

圖片網址: cats_QQ.jpg
圖片網址: cats_Mi.jpg
超連結網址: https://www.catsdate.com/
超連結網址: https://www.pettalk.tw/


##### 13.4.3 取得完整的超連結URL

In [4]:
# ch13_4.py, 取得圖片完整的URL
import requests
from bs4 import BeautifulSoup
url = 'https://wienhong.github.io/cats.html'
base_url = 'https://wienhong.github.io/'
req = requests.get(url)
req.encoding = 'utf-8'
sp = BeautifulSoup(req.text, 'html.parser')

# 找出所有的<img>標籤
images = sp.find_all('img')     		# 找出所有的<img>標籤
for img in images:
    img_url = img.get('src')    		# 取得圖片的 URL
    if not img_url.startswith('http'):  # 若不是完整的URL
        img_url = base_url + img_url  	# 加上網域名稱
    print(f'圖片網址: {img_url}')

圖片網址: https://wienhong.github.io/cats_QQ.jpg
圖片網址: https://wienhong.github.io/cats_Mi.jpg


#### 13.5 爬取 PTT 寵物板

##### 13.5.1 爬取一個頁面的資訊

In [5]:
# ch13_5.py, 取得文章標題
import requests
from bs4 import BeautifulSoup
url = 'https://www.ptt.cc/bbs/pet/index.html'
req = requests.get(url)
req.encoding = 'utf-8'
sp = BeautifulSoup(req.text, 'html.parser')

articles = sp.find_all('div', class_='r-ent')# 找出所有的文章區塊
for data in articles:
    try:
        title_block = data.find('div', class_='title')  # 找標題
        title_tag = title_block.find('a')
        title = title_tag.text.strip()
    except:
        title = '本文已被刪除'
    print(title)

[問題] 狗狗要被列管為攻擊性犬隻
[交易] 賣 高雄 neakasa M1 自動貓沙機
[交易] 售 UG home 全新 3尺 白鐵狗籠
[交易] PUBT 寵物推車
Re: [資訊] 寵物狗貓交流Line群組
[交易] AiRROBO C10 Pro 自動貓砂機
[贈送] 希爾斯~成貓7歲以上主食罐*11
[閒聊] 陪我15年的狗狗離開了。。。
[問題] 明知道他會比你早離開為什麼要養
Re: [問題] 明知道他會比你早離開為什麼要養
[問題] 想詢問養兔子的經驗
贈送寵物用品
Re: [問題] 轉錄文章無法刪除
[事件] 網路社群，小心詐騙!
[公告] 寵物板板規
[公告] 關於有些文章無法自行刪除
[公告] 防詐宣導


In [6]:
# ch13_6.py, 取得文章標題、作者、發文日期
import requests
from bs4 import BeautifulSoup
url = 'https://www.ptt.cc/bbs/pet/index.html'
req = requests.get(url)
req.encoding = 'utf-8'
sp = BeautifulSoup(req.text, 'html.parser')

articles = sp.find_all('div', class_='r-ent')# 找出所有的文章區塊
for data in articles:
    try:
        title_block = data.find('div', class_='title')  # 找標題
        title_tag =title_block.find('a')     # 找出<a>標籤                           
        title = title_tag.text.strip()                        
        meta_tag=data.find('div',class_='meta')# 找作者和發文日期
        author =meta_tag.find('div', class_='author').text            
        date=meta_tag.find('div', class_='date').text.strip()
        print(f'{title}, {author}, {date}')   # 列印結果
    except:
        title = '本文已被刪除'
        print(title)   	                     # 列印結果

[問題] 狗狗要被列管為攻擊性犬隻, juliepretty, 4/25
[交易] 賣 高雄 neakasa M1 自動貓沙機, tom38921, 4/27
[交易] 售 UG home 全新 3尺 白鐵狗籠, titik, 5/08
[交易] PUBT 寵物推車, miller78931, 5/08
Re: [資訊] 寵物狗貓交流Line群組, padmafeel, 5/08
[交易] AiRROBO C10 Pro 自動貓砂機, mikes90119, 5/10
[贈送] 希爾斯~成貓7歲以上主食罐*11, setsunaxia, 5/10
[閒聊] 陪我15年的狗狗離開了。。。, sailorsakura, 5/12
[問題] 明知道他會比你早離開為什麼要養, landibra, 5/14
Re: [問題] 明知道他會比你早離開為什麼要養, kiwiuuuwant, 5/15
[問題] 想詢問養兔子的經驗, mewjudy, 5/15
贈送寵物用品, abliu1, 5/20
Re: [問題] 轉錄文章無法刪除, energyshine, 8/26
[事件] 網路社群，小心詐騙!, yuyu1130, 9/24
[公告] 寵物板板規, Rakitic, 4/07
[公告] 關於有些文章無法自行刪除, Rakitic, 4/07
[公告] 防詐宣導, Rakitic, 11/05


##### 13.5.2 爬取多個頁面

In [8]:
# ch13_7.py, 取得5頁文章標題
import requests
from bs4 import BeautifulSoup
base_url = 'https://www.ptt.cc'     # PTT主網域名稱
current_url = '/bbs/pet/index.html' # 當前頁面的路徑
with open('ch13_7.txt', 'w', encoding='utf-8') as f:    
    n_pages = 5     # 要爬取的頁數
    for i in range(n_pages):
        f.write(f'第{i+1}頁\n')
        url = base_url + current_url  # 取得當前頁面的完整網址
        req = requests.get(url)
        sp = BeautifulSoup(req.text, 'html.parser')
        # 找出所有的文章區塊 
        articles = sp.find_all('div',class_='r-ent')   
        for data in articles:
            try:  		
                title_block = data.find('div', class_='title') 
                title_tag =title_block.find('a') 	# 找<a>標籤
                title = title_tag.text.strip()    # 標題
            except:
                title = '本文已被刪除'
            f.write(f"{title}\n")  # 寫入檔案  
        # 找出上頁連結
        btn_group = sp.find('div', class_='btn-group-paging')
        if btn_group:    	 # 如果有找到按鈕群組
            prev_page = btn_group.find_all('a')[1]
            if prev_page: # 如果有找到上一頁按鈕
                current_url = prev_page['href'] # 更新current_url
            else:
                break
        else:
            break
print("爬取結果已存入檔案中")


爬取結果已存入檔案中


In [9]:
# ch13_8.py, 取得5頁文章標題、連結
import requests
from bs4 import BeautifulSoup
base_url = 'https://www.ptt.cc'     # PTT主網域名稱
current_url = '/bbs/pet/index.html' # 當前頁面的路徑
with open('ch13_8.txt', 'w', encoding='utf-8') as f:
    f.write('文章標題, 連結\n')    
    n_pages = 5     # 要爬取的頁數
    for i in range(n_pages):
        f.write(f'第{i+1}頁\n')
        url = base_url + current_url  # 取得當前頁面的完整網址
        req = requests.get(url)
        sp = BeautifulSoup(req.text, 'html.parser')
        # 找出所有的文章區塊
        articles = sp.find_all('div',class_='r-ent')        
        for data in articles:
            try:
                title_block = data.find('div', class_='title') 
                title_tag =title_block.find('a')	# 找<a>標籤
                title = title_tag.text.strip()  	# 找標題
                link = base_url + title_tag['href'] 	# 文章連結                
                f.write(f"{title}, {link}\n")       	# 寫入檔案
            except:
                title = '本文已被刪除'
                f.write(f"{title}\n")       # 寫入檔案
        # 找出上頁連結
        btn_group = sp.find('div', class_='btn-group-paging')
        if btn_group:    # 如果有找到按鈕群組
            prev_page = btn_group.find_all('a')[1]
            if prev_page: # 如果有找到上一頁按鈕
                current_url = prev_page['href'] # 更新current_url
            else:
                break
        else:
            break
print("爬取結果已存入檔案中")


爬取結果已存入檔案中
